# parameter-wrap-around-tensor — ex2: fix WrapParam anti-pattern by converting to IsAParam in-place

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `parameter-wrap-around-tensor`. Running the final beacon cell reports progress against the `Backprop: Parameter wrap around Tensor` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Parameter wrap around Tensor` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`parameter-wrap-around-tensor`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "parameter-wrap-around-tensor"
DD_SUBTOPIC = "Backprop: Parameter wrap around Tensor"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Repairing the composition-Parameter anti-pattern — deepening

Ex1 observed that `WrapParam(tensor)` (composition / HAS-A) is silently invisible to `isinstance(_, MiniTensor)` filters. The fix is to convert each WrapParam to an `IsAParam(MiniTensor)` subclass instance BEFORE the autograd layer ever sees it.

```python
def fix_params(things):
    fixed = []
    for p in things:
        if isinstance(p, WrapParam):
            fixed.append(IsAParam(p.tensor))
        else:
            fixed.append(p)
    return fixed
```

**Why a conversion helper and not a class-rewrite.** In a real codebase you might inherit a config that uses the WrapParam form; you can't rewrite the class without breaking other consumers. A single 'normalize before training' pass is the safest fix.

**Round-trip the fix.** After the conversion, every parameter passes `isinstance(_, MiniTensor)` — the autograd layer collects them all, the SGD step updates them all, the bug is gone.

### Exercise 2 — fix WrapParam anti-pattern by converting to IsAParam in-place

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply a conversion helper that maps `WrapParam` (composition) instances to `IsAParam` (subclass) instances while preserving tensor identity, then verify the round-trip survives the autograd-layer `isinstance` filter and a fake optimizer's update.
> Keywords: parameter, fix, conversion, isinstance, anti-pattern-repair
> ```

**KCs targeted:** `parameter-wrap-around-tensor`, `parameter-subclass-of-tensor`

Implement `fix_params(things)` and `fake_optimizer_step(params, lr)` together. The drill exercises the REPAIR side of the anti-pattern from ex1.

Definitions you may reference (defined in the stub for you):
- `WrapParam(tensor)` — composition Parameter. Stores `.tensor`. NOT a MiniTensor subclass.
- `IsAParam(MiniTensor)` — subclass Parameter. Inherits.

**1. `fix_params(things)` — list -> list.**
Walk `things`. For each element:
- If it's a `WrapParam`, build a new `IsAParam(p.tensor)` and include that.
- Otherwise (already IsAParam, plain MiniTensor, or unrelated junk), include it unchanged.
Return a NEW list. Don't mutate the input.

**2. `fake_optimizer_step(params, lr)`.**
Simulate what a real optimizer does to a parameter list:
- Filter `params` with `isinstance(_, MiniTensor)`.
- For each survivor, decrement `p.array` in place by `lr * t.ones_like(p.array)` (we don't have real gradients here — this stands in for a uniform 'step away from zero' move so the test can detect which params got updated).
- Return the count of survivors (also = number of params that got updated).

**Why fake_optimizer_step.** It's the minimal model of any autograd-layer helper. If `WrapParam`s are silently dropped, the count is wrong AND the `.tensor` stays unchanged. After `fix_params`, the count is right AND every `.array` shifts.

In [ ]:
class WrapParam:
    """Composition Parameter (anti-pattern from ex1)."""
    def __init__(self, tensor):
        self.tensor = tensor
        self.requires_grad = True


class IsAParam(MiniTensor):
    """Subclass Parameter (correct design)."""
    def __init__(self, array):
        super().__init__(array, requires_grad=True)


def fix_params(things: list) -> list:
    """Convert WrapParams to IsAParams, leave others untouched."""
    raise NotImplementedError()


def fake_optimizer_step(params: list, lr: float) -> int:
    """Filter by isinstance(MiniTensor), apply uniform step, return count updated."""
    raise NotImplementedError()


def _test_ex2():
    # --- invariant 1: WITHOUT the fix, fake_optimizer skips WrapParams ---
    t.manual_seed(0)
    raw_tensors = [t.tensor([1.0, 2.0]), t.tensor([3.0, 4.0]), t.tensor([5.0, 6.0])]
    mixed = [WrapParam(raw_tensors[0]), WrapParam(raw_tensors[1]), IsAParam(raw_tensors[2].clone())]
    n_before = fake_optimizer_step(mixed, lr=0.1)
    assert n_before == 1, (
        f'WITHOUT fix: only the IsAParam should be updated; expected 1, got {n_before}'
    )
    # The two WrapParams' tensors are unchanged (the bug):
    assert t.allclose(mixed[0].tensor, t.tensor([1.0, 2.0])), 'WrapParam 0 silently skipped'
    assert t.allclose(mixed[1].tensor, t.tensor([3.0, 4.0])), 'WrapParam 1 silently skipped'

    # --- invariant 2: WITH the fix, all three get updated ---
    mixed2 = [WrapParam(t.tensor([1.0, 2.0])), WrapParam(t.tensor([3.0, 4.0])),
              IsAParam(t.tensor([5.0, 6.0]))]
    fixed = fix_params(mixed2)
    assert len(fixed) == 3, f'fix_params must preserve list length, got {len(fixed)}'
    # Every element is now a MiniTensor instance.
    assert all(isinstance(p, MiniTensor) for p in fixed), (
        f'every fixed element must pass isinstance(MiniTensor); got {[type(p).__name__ for p in fixed]}'
        )
    n_after = fake_optimizer_step(fixed, lr=0.1)
    assert n_after == 3, f'after fix: all 3 must update; expected 3, got {n_after}'
    # Each .array is now shifted by lr (vector of ones), -> subtract 0.1
    for i, p in enumerate(fixed):
        diff = p.array - t.tensor([1.0 + 2*i, 2.0 + 2*i])
        # We subtracted 0.1 from each element, so diff == -0.1
        assert t.allclose(diff, t.full_like(diff, -0.1), atol=1e-6), (
            f'param {i} expected shift of -0.1 per element, got {diff}'
        )

    # --- invariant 3: input list is not mutated ---
    src = [WrapParam(t.tensor([7.0])), IsAParam(t.tensor([8.0]))]
    snapshot_types = [type(p).__name__ for p in src]
    out = fix_params(src)
    assert [type(p).__name__ for p in src] == snapshot_types, (
        f'fix_params must NOT mutate the input list: was {snapshot_types}, now {[type(p).__name__ for p in src]}'
    )
    assert out is not src, 'fix_params must return a new list, not the same one'

    # --- invariant 4: tensor identity preserved through conversion ---
    raw = t.tensor([42.0, 43.0])
    wp = WrapParam(raw)
    [fixed_one] = fix_params([wp])
    assert isinstance(fixed_one, IsAParam)
    assert fixed_one.array is raw, 'converting should reuse the underlying tensor, not copy'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
class WrapParam:
    def __init__(self, tensor):
        self.tensor = tensor
        self.requires_grad = True


class IsAParam(MiniTensor):
    def __init__(self, array):
        super().__init__(array, requires_grad=True)


def fix_params(things: list) -> list:
    out = []
    for p in things:
        if isinstance(p, WrapParam):
            out.append(IsAParam(p.tensor))
        else:
            out.append(p)
    return out


def fake_optimizer_step(params: list, lr: float) -> int:
    count = 0
    for p in params:
        if isinstance(p, MiniTensor):
            p.array -= lr * t.ones_like(p.array)
            count += 1
    return count
```

**Reuse the underlying tensor on conversion.** `IsAParam(p.tensor)` hands the SAME `torch.Tensor` object into the new wrapper. Any external reference to the raw tensor still sees the in-place updates from the optimizer. Copying would duplicate memory AND break those references.

**Why `fake_optimizer_step` returns the count.** It's the smallest observable that proves the filter worked. Without `fix_params`, count=1 (silent two-thirds drop). With it, count=3 (every param survived). Real bug-hunting in a codebase often starts with exactly this — print `len(list(model.parameters()))` and notice the number is wrong.

**The lesson generalizes.** Anywhere `isinstance(_, Base)` gates behavior, COMPOSITION ('I hold a Base') invisibly fails while INHERITANCE ('I AM a Base') passes. The same pattern appears in framework hooks, plugin registries, and abstract type-class dispatch.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()